# Sprint 2 - all candidate models, one protocol

- `E.setup(seed=42)` grouped holdout -> `E.dedup_configs` on the train fold -> consensus targets -> `E.score`
- Metric: EMR, exact match over all 731 BIL columns; threshold 0.5; fit on unique configurations, unweighted
- Every model is refitted here, so all EMRs are directly comparable

| model | description |
|---|---|
| `lookup-majority` | exact CRM-config lookup, global modal BIL fallback |
| `ovr-l1-lr` | one-vs-rest L1 logistic regression + additive rules |
| `chain-l1-lr` | cluster-ordered classifier chain, L1 LR + additive rules |
| `br-lgbm` | binary relevance LightGBM, `mcs=20`, `reg_lambda=1` |
| `br-lgbm-rules` | same fit + additive rules |
| `wide-deep-512` | wide additive path + shared 512-unit hidden layer |
| `neighbour-delta` | nearest CRM config + per-pack BIL delta |
| `blend-wd-nd-lgbm[-rules]` | equal average of `wide-deep-512`, `neighbour-delta`, `br-lgbm` |

## Setup

In [1]:
import os
import sys
import warnings

sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import networkx as nx
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.stats import binomtest
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import ClassifierChain

from src import data, experiment as E, metrics as M
from src.models import BinaryRelevanceLGBM, CoOccurrenceChainLGBM
from src.models_nndelta import NeighbourDelta
from src.models_widedeep import WideDeepMultiLabel

SEED = 42
THR = 0.5
N_BOOT = 2000
CACHE_DIR = os.path.join(data.DATA_DIR, 'cache')

d = E.setup(seed=SEED, val_size=0.2)
tr, va = d['tr'], d['va']
X_tr, Y_tr, _multiplicity = E.dedup_configs(d['X_train'][tr], d['Y_cons'][tr])
X_va, Y_va = d['X_train'][va], d['Y_cons'][va]
BIL = list(d['bil_cols'])

# configurations are the independent unit for the paired tests; size = rows per config
cfg, cfg_first, cfg_size = np.unique(d['groups'][va], return_index=True, return_counts=True)
w = cfg_size.astype(np.float64)

# the split is grouped on CRM configuration, so no validation configuration is
# available to a lookup model: this is what caps the lookup baseline, by construction
shared = np.intersect1d(np.unique(d['groups'][tr]), np.unique(d['groups'][va]))
print(f'configurations shared between train and val: {len(shared)}')

print(f'train {len(X_tr):,} unique configs x {X_tr.shape[1]} CRM columns')
print(f'val   {len(X_va):,} rows / {len(cfg):,} configs x {Y_tr.shape[1]} BIL columns')

configurations shared between train and val: 0
train 53,947 unique configs x 745 CRM columns
val   36,385 rows / 13,487 configs x 731 BIL columns


## Models

- The lookup, one-vs-rest and chain models are wrapped inline, so this notebook is self-contained
- Chain order: `near_duplicate_bil_pairs.csv` at Jaccard >= 0.90 -> connected components -> clusters sorted by their most frequent label

In [2]:
def with_rules(pred):
    """Sprint 1 additive rules on the validation rows."""
    return data.apply_additive_rules(pred, X_va, d['crm_cols'], d['bil_cols'])


class LookupMajority:
    """Tier A: exact CRM-configuration lookup with a global modal BIL fallback."""

    def fit(self, X, Y):
        keys = np.packbits(np.asarray(X, np.uint8), axis=1)
        self.table_ = {k.tobytes(): i for i, k in enumerate(keys)}
        self.Y_ = np.asarray(Y, np.uint8)
        bkeys = [r.tobytes() for r in np.packbits(self.Y_, axis=1)]
        modal = pd.Series(bkeys).value_counts().index[0]
        self.fallback_ = self.Y_[bkeys.index(modal)]
        return self

    def predict_proba(self, X):
        keys = np.packbits(np.asarray(X, np.uint8), axis=1)
        out = np.empty((len(X), self.Y_.shape[1]), np.float32)
        self.n_hit_ = 0
        for i, k in enumerate(keys):
            j = self.table_.get(k.tobytes())
            out[i] = self.fallback_ if j is None else self.Y_[j]
            self.n_hit_ += j is not None
        print(f'    lookup hits {self.n_hit_:,}/{len(X):,} validation rows')
        return out


class OvRL1LR:
    """Tier A: one-vs-rest L1 logistic regression."""

    def fit(self, X, Y):
        base = LogisticRegression(penalty='l1', solver='liblinear', C=1.0, random_state=SEED)
        self.m_ = OneVsRestClassifier(base, n_jobs=-1).fit(csr_matrix(X), Y)
        return self

    def predict_proba(self, X):
        return np.asarray(self.m_.predict_proba(csr_matrix(X)), dtype=np.float32)


def cluster_chain_order(Y, bil_cols, threshold=0.90):
    """Chain order: near-duplicate label clusters, most frequent first."""
    pairs = pd.read_csv(os.path.join(data.DERIVED_DIR, 'near_duplicate_bil_pairs.csv'))
    strong = pairs[pairs['jaccard'] >= threshold]
    freq = dict(zip(bil_cols, np.asarray(Y).mean(0)))
    G = nx.Graph()
    G.add_nodes_from(bil_cols)
    for _, r in strong.iterrows():
        G.add_edge(r['col_a'], r['col_b'], weight=r['jaccard'])
    clusters = sorted(nx.connected_components(G),
                      key=lambda c: max(freq[n] for n in c), reverse=True)
    order_labels = []
    for c in clusters:
        order_labels += sorted(c, key=lambda n: freq[n], reverse=True)
    idx = {c: i for i, c in enumerate(bil_cols)}
    return [idx[c] for c in order_labels], len(clusters)


class ClusterChainLR:
    """Tier B: classifier chain over the cluster order, L1 LR base."""

    def fit(self, X, Y):
        order, self.n_clusters_ = cluster_chain_order(Y, BIL)
        print(f'    chain order over {self.n_clusters_} label clusters')
        base = LogisticRegression(penalty='l1', solver='liblinear', C=1.0, random_state=SEED)
        self.m_ = ClassifierChain(base, order=order, random_state=SEED).fit(csr_matrix(X), Y)
        return self

    def predict_proba(self, X):
        # sklearn returns columns in the original label order, not the chain order
        return np.asarray(self.m_.predict_proba(csr_matrix(X)), dtype=np.float32)


CHAMP = dict(min_child_samples=20, reg_lambda=1.0)
WD = dict(hidden=(512,), dropout=0.2, wide=True, lr=2e-3,
          batch_size=256, max_epochs=80, patience=8)
NND = dict(min_pairs=2, min_rate=0.5, max_nb=5)

# one fit per key; models sharing a fit (br-lgbm with/without rules) reuse its probabilities
FITS = {
    'lookup-majority': LookupMajority,
    'ovr-l1-lr': OvRL1LR,
    'chain-l1-lr': ClusterChainLR,
    'br-lgbm': lambda: BinaryRelevanceLGBM(CHAMP),
    'wide-deep-512': lambda: WideDeepMultiLabel(seed=SEED, verbose=False, **WD),
    'neighbour-delta': lambda: NeighbourDelta(**NND),
    'cooc-chain-lgbm': lambda: CoOccurrenceChainLGBM(CHAMP, n_parents=30),
}

# model id -> (fit key, apply rules, description)
MODELS = [
    ('lookup-majority', 'lookup-majority', False,
     'exact CRM-config lookup, global modal BIL fallback'),
    ('ovr-l1-lr', 'ovr-l1-lr', True,
     'one-vs-rest L1 logistic regression + additive rules'),
    ('chain-l1-lr', 'chain-l1-lr', True,
     'cluster-ordered classifier chain, L1 LR base + additive rules'),
    ('br-lgbm', 'br-lgbm', False,
     'binary relevance LightGBM, min_child_samples=20, reg_lambda=1'),
    ('br-lgbm-rules', 'br-lgbm', True,
     'same fit, with the additive rules applied'),
    ('wide-deep-512', 'wide-deep-512', False,
     'wide additive path + shared 512-unit hidden layer, BCE'),
    ('neighbour-delta', 'neighbour-delta', False,
     'nearest CRM config, then the known per-pack BIL delta'),
    ('cooc-chain-lgbm', 'cooc-chain-lgbm', False,
     'co-occurrence-ordered classifier chain, LightGBM base, 30 parents'),
    ('cooc-chain-lgbm-rules', 'cooc-chain-lgbm', True,
     'same fit, with the additive rules applied'),
]

BLEND_MEMBERS = ['wide-deep-512', 'neighbour-delta', 'br-lgbm']
print(f'{len(FITS)} fits -> {len(MODELS)} scored models + 2 blends')

7 fits -> 9 scored models + 2 blends


## Fit

- Probabilities cached to `data/cache/`; a re-run reloads instead of refitting

In [3]:
def fit_or_load(key):
    f = os.path.join(CACHE_DIR, f'merged_proba_{key}_s{SEED}.npy')
    if os.path.exists(f):
        print(f'  [{key}] cached')
        return np.load(f), 0.0
    print(f'  [{key}] fitting...', flush=True)
    with E.Timer() as t:
        P = np.asarray(FITS[key]().fit(X_tr, Y_tr).predict_proba(X_va), dtype=np.float32)
    os.makedirs(CACHE_DIR, exist_ok=True)
    np.save(f, P)
    print(f'  [{key}] done in {t.s:.0f}s', flush=True)
    return P, t.s


# cheapest first; each fit is cached on success, so an interrupted run resumes
FIT_ORDER = ['lookup-majority', 'neighbour-delta', 'wide-deep-512',
             'cooc-chain-lgbm', 'br-lgbm', 'ovr-l1-lr', 'chain-l1-lr']
assert set(FIT_ORDER) == set(FITS), 'FIT_ORDER must list every fit in FITS'

proba, secs = {}, {}
for key in FIT_ORDER:
    proba[key], secs[key] = fit_or_load(key)

assert len(proba) == len(FITS), 'every model must fit'
print(f'\nfitted {len(proba)}/{len(FITS)}')

  [lookup-majority] cached
  [neighbour-delta] cached
  [wide-deep-512] cached
  [cooc-chain-lgbm] cached
  [br-lgbm] cached


  [ovr-l1-lr] cached


  [chain-l1-lr] cached

fitted 7/7


## Score

In [4]:
rows, ok = [], {}


def record(model_id, P, rules, desc, train_s):
    pred = (P >= THR).astype(np.uint8)
    if rules:
        pred = with_rules(pred)
    res = E.score(d, pred)
    ok[model_id] = ((pred != Y_va).sum(1) == 0)[cfg_first]
    rows.append(dict(model=model_id, rules=rules, train_s=round(train_s, 1),
                     description=desc, **res))
    print(f'{model_id:<24} EMR {res["emr"]:.5f}  HL {res["hamming_loss"]:.6f}')


for model_id, key, rules, desc in MODELS:
    record(model_id, proba[key], rules, desc, secs[key])

P_blend = np.mean([proba[m] for m in BLEND_MEMBERS], axis=0)
record('blend-wd-nd-lgbm', P_blend, False,
       'equal average of wide-deep-512, neighbour-delta, br-lgbm', 0.0)
record('blend-wd-nd-lgbm-rules', P_blend, True,
       'same blend, with the additive rules applied', 0.0)

lookup-majority          EMR 0.00016  HL 0.023358


ovr-l1-lr                EMR 0.86437  HL 0.001588


chain-l1-lr              EMR 0.86816  HL 0.001610
br-lgbm                  EMR 0.87654  HL 0.001327


br-lgbm-rules            EMR 0.87841  HL 0.001375
wide-deep-512            EMR 0.87926  HL 0.002328


neighbour-delta          EMR 0.81495  HL 0.002060
cooc-chain-lgbm          EMR 0.87514  HL 0.001327


cooc-chain-lgbm-rules    EMR 0.87709  HL 0.001375


blend-wd-nd-lgbm         EMR 0.88264  HL 0.001364
blend-wd-nd-lgbm-rules   EMR 0.88286  HL 0.001406


## Results

In [5]:
tbl = pd.DataFrame(rows).sort_values('emr', ascending=False).reset_index(drop=True)
cols = ['model', 'rules', 'emr', 'emr_vs_raw', 'hamming_loss',
        'share_dist1', 'share_dist2', 'share_dist3plus', 'train_s']
display(tbl[cols].style.format({'emr': '{:.4%}', 'emr_vs_raw': '{:.4%}',
                                'hamming_loss': '{:.6f}', 'share_dist1': '{:.3%}',
                                'share_dist2': '{:.3%}', 'share_dist3plus': '{:.3%}'}))

best = tbl.iloc[0]['model']
print(f'best: {best} at {tbl.iloc[0]["emr"]:.4%}')
display(tbl[['model', 'description']])

,model,rules,emr,emr_vs_raw,hamming_loss,share_dist1,share_dist2,share_dist3plus,train_s
0,blend-wd-nd-lgbm-rules,True,88.2864%,87.8741%,0.001406,1.385%,0.808%,9.520%,0.000000
1,blend-wd-nd-lgbm,False,88.2644%,87.8521%,0.001364,1.426%,0.827%,9.482%,0.000000
2,wide-deep-512,False,87.9263%,87.5306%,0.002328,1.487%,0.709%,9.878%,0.000000
3,br-lgbm-rules,True,87.8411%,87.3547%,0.001375,1.874%,0.833%,9.452%,0.000000
4,cooc-chain-lgbm-rules,True,87.7092%,87.2228%,0.001375,1.874%,0.962%,9.454%,0.000000
5,br-lgbm,False,87.6543%,87.1678%,0.001327,2.081%,0.860%,9.405%,0.000000
6,cooc-chain-lgbm,False,87.5141%,87.0276%,0.001327,2.083%,0.998%,9.405%,0.000000
7,chain-l1-lr,True,86.8160%,86.3405%,0.001610,2.229%,1.187%,9.768%,0.000000
8,ovr-l1-lr,True,86.4367%,86.0052%,0.001588,2.619%,1.176%,9.768%,0.000000
9,neighbour-delta,False,81.4951%,81.1433%,0.002060,6.984%,1.011%,10.510%,0.000000


best: blend-wd-nd-lgbm-rules at 88.2864%


,model,description
0,blend-wd-nd-lgbm-rules,"same blend, with the additive rules applied"
1,blend-wd-nd-lgbm,"equal average of wide-deep-512, neighbour-delt..."
2,wide-deep-512,wide additive path + shared 512-unit hidden la...
3,br-lgbm-rules,"same fit, with the additive rules applied"
4,cooc-chain-lgbm-rules,"same fit, with the additive rules applied"
5,br-lgbm,"binary relevance LightGBM, min_child_samples=2..."
6,cooc-chain-lgbm,"co-occurrence-ordered classifier chain, LightG..."
7,chain-l1-lr,"cluster-ordered classifier chain, L1 LR base +..."
8,ovr-l1-lr,one-vs-rest L1 logistic regression + additive ...
9,neighbour-delta,"nearest CRM config, then the known per-pack BI..."


## Paired tests against the best model

- Unit is the CRM configuration, not rows; CI is a cluster bootstrap on the row-weighted EMR difference
- A CI excluding zero is the claim

In [6]:
rng = np.random.default_rng(0)
ref = ok[best]
sig = []
for model_id, o in ok.items():
    if model_id == best:
        continue
    b, c = int((o & ~ref).sum()), int((~o & ref).sum())
    diff = float((w * (o.astype(float) - ref)).sum() / w.sum())
    dv = (o.astype(float) - ref) * w
    boots = []
    for _ in range(0, N_BOOT, 200):
        pw = rng.poisson(1.0, size=(200, len(w)))
        boots.extend((pw @ dv) / (pw @ w))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    sig.append(dict(model=model_id, vs=best, dEMR_pts=diff * 100,
                    ci_lo=lo * 100, ci_hi=hi * 100, only_this_right=b, only_ref_right=c,
                    mcnemar_p=binomtest(b, b + c, 0.5).pvalue if b + c else 1.0,
                    significant='yes' if (lo > 0 or hi < 0) else 'no'))

sg = pd.DataFrame(sig).sort_values('dEMR_pts', ascending=False).reset_index(drop=True)
display(sg.style.format({'dEMR_pts': '{:+.3f}', 'ci_lo': '{:+.3f}', 'ci_hi': '{:+.3f}',
                         'mcnemar_p': '{:.3g}'}))

tbl.to_csv(os.path.join(data.DERIVED_DIR, 'sprint2_all_models_comparison.csv'), index=False)
sg.to_csv(os.path.join(data.DERIVED_DIR, 'sprint2_all_models_significance.csv'), index=False)
print('written -> data/derived/sprint2_all_models_{comparison,significance}.csv')

,model,vs,dEMR_pts,ci_lo,ci_hi,only_this_right,only_ref_right,mcnemar_p,significant
0,blend-wd-nd-lgbm,blend-wd-nd-lgbm-rules,-0.022,-0.048,+0.000,5,13,0.0963,no
1,wide-deep-512,blend-wd-nd-lgbm-rules,-0.360,-0.575,-0.209,59,152,1.2e-10,yes
2,br-lgbm-rules,blend-wd-nd-lgbm-rules,-0.445,-0.741,-0.159,55,187,5.19e-18,yes
3,cooc-chain-lgbm-rules,blend-wd-nd-lgbm-rules,-0.577,-0.970,-0.225,54,191,3.96e-19,yes
4,br-lgbm,blend-wd-nd-lgbm-rules,-0.632,-0.949,-0.327,61,260,2.13e-30,yes
5,cooc-chain-lgbm,blend-wd-nd-lgbm-rules,-0.772,-1.201,-0.398,60,267,2.62e-32,yes
6,chain-l1-lr,blend-wd-nd-lgbm-rules,-1.470,-1.973,-1.067,23,431,1.29e-98,yes
7,ovr-l1-lr,blend-wd-nd-lgbm-rules,-1.850,-2.452,-1.362,23,519,2.67e-123,yes
8,neighbour-delta,blend-wd-nd-lgbm-rules,-6.791,-14.811,-1.718,58,491,1.81e-86,yes
9,lookup-majority,blend-wd-nd-lgbm-rules,-88.270,-90.393,-86.037,6,9402,0,yes


written -> data/derived/sprint2_all_models_{comparison,significance}.csv


## Lookup baseline: why the published test-set number differs

- The reported **EMR 0.000072** for this family was measured on the **test set against `solution.csv`**; everything above is validation EMR, so the two are not comparable
- The cause carries over: the grouped split shares **zero** CRM configurations between train and validation (printed in the setup cell), so an exact-lookup model has nothing to match and always falls back to the global mode. Hence **0.0165%** here
- That makes the result structural rather than a tuning problem, which is what the source notebook concluded from varying the relabelling threshold: EMR stayed frozen regardless
- Only the base lookup is reproduced here, not the threshold sweep, since the reported sweep moved EMR by at most 1 row in 97,100
- The original inline evaluator is kept below and cross-checked against `src/metrics.py`

In [7]:
from sklearn.metrics import f1_score, roc_auc_score


def reconstruct_full_predictions(y_model_pred_df, test_features_df, bil_to_crm_triggers,
                                 all_bil_cols):
    """Recombines model predictions on Y_model with deterministic rules on Y_additive."""
    additive_dict = {}
    for b_pack, c_packs in bil_to_crm_triggers.items():
        avail_c = [c for c in c_packs if c in test_features_df.columns]
        if avail_c:
            additive_dict[b_pack] = test_features_df[avail_c].max(axis=1).values
        else:
            additive_dict[b_pack] = np.zeros(len(test_features_df), dtype=np.int8)
    additive_df = pd.DataFrame(additive_dict, index=test_features_df.index)
    full_preds = pd.concat([y_model_pred_df, additive_df], axis=1)
    return full_preds.reindex(columns=all_bil_cols, fill_value=0)


def evaluate_multilabel_performance(y_true, y_pred_prob, threshold=0.5):
    """Macro/Micro ROC-AUC, Micro/Macro F1, and Exact Match Ratio."""
    y_true_np = np.asarray(y_true)
    y_pred_bin = (np.asarray(y_pred_prob) >= threshold).astype(int)
    metrics = {}
    try:
        metrics['roc_auc_micro'] = roc_auc_score(y_true_np, y_pred_prob, average='micro')
        metrics['roc_auc_macro'] = roc_auc_score(y_true_np, y_pred_prob, average='macro')
    except Exception as e:
        metrics['roc_auc_note'] = str(e)
    metrics['f1_micro'] = f1_score(y_true_np, y_pred_bin, average='micro')
    metrics['f1_macro'] = f1_score(y_true_np, y_pred_bin, average='macro')
    metrics['exact_match_ratio'] = np.all(y_true_np == y_pred_bin, axis=1).mean()
    return metrics


# cross-check: the inline EMR definition against src/metrics.py, on the best model
model_to_fit = {m: k for m, k, _, _ in MODELS}
P_chk = proba[model_to_fit[best]] if best in model_to_fit else P_blend
inline = evaluate_multilabel_performance(Y_va, P_chk, THR)
ours = M.exact_match_ratio(Y_va, (P_chk >= THR).astype(np.uint8))
print(f"inline evaluator EMR {inline['exact_match_ratio']:.6f} | "
      f"src/metrics exact_match_ratio {ours:.6f} | "
      f"identical: {abs(inline['exact_match_ratio'] - ours) < 1e-12}")
print(f"F1 micro {inline['f1_micro']:.4f} | F1 macro {inline['f1_macro']:.4f}")

inline evaluator EMR 0.882644 | src/metrics exact_match_ratio 0.882644 | identical: True
F1 micro 0.9661 | F1 macro 0.3804


## Tier C CatBoost - code included, not run here

- Source: commit `2395f52`, `notebooks/sprint2_tierC.ipynb`
- **Not executed in this notebook.** The reported fit took 8422.9 s (2 h 20 min), against
  321 s for the slowest model above, so it is reproduced as code rather than re-run
- The reported figures are EMR **0.8304246** and Hamming loss **0.0014544**, which place it
  below `br-lgbm` on both (87.65% EMR, 0.001327 Hamming)
- Recorded conclusion: global `MultiCrossEntropy` squashes rare labels in favour of Hamming
  loss, with median F1 collapsing to 0 - an argument for binary relevance over a
  globally-pooled multi-label loss
- The source cells carry no saved outputs either, so the figures above come from the team
  tracker row, not from a stored run

```python
from catboost import CatBoostClassifier

d = E.setup(seed=42)
tr, va = d['tr'], d['va']
Xa_dense, Ya, cnt = E.dedup_configs(d['X_train'][tr], d['Y_cons'][tr])
X_val_dense = d['X_train'][va]
Yv = d['Y_cons'][va]

cb_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    loss_function='MultiCrossEntropy',
    custom_metric=['HammingLoss'],
    random_seed=42,
    verbose=100,
)

with E.Timer() as t:
    cb_model.fit(Xa_dense, Ya)

P_val = cb_model.predict_proba(X_val_dense)
pred_val = (P_val >= 0.5).astype(np.uint8)
pred_val_final = data.apply_additive_rules(
    pred_val, d['X_train'][va], d['crm_cols'], d['bil_cols'])

val_summary, val_cm, val_f1 = M.multilabel_report(Yv, pred_val_final, P_val)
```

To include it in the comparison above, add it to `FITS` as a wrapper exposing
`fit(X, Y)` / `predict_proba(X)` and to `MODELS` with `rules=True`.

## Conclusions

- Ranking and significance are the two tables above; also written to `data/derived/sprint2_all_models_{comparison,significance}.csv`
- Champion is `blend-wd-nd-lgbm-rules` at **88.29%**. Its edge over `blend-wd-nd-lgbm` (+0.022 pts, CI -0.047 to 0.000, p = 0.10) is **not significant** - 5 configurations one way, 13 the other. Every other model is significantly below it, all CIs excluding zero
- All previously published figures reproduce exactly under this single protocol: `ovr-l1-lr` 86.4367, `chain-l1-lr` 86.8160, `br-lgbm` 87.6543, `cooc-chain-lgbm-rules` 87.7092, `br-lgbm-rules` 87.8411, `wide-deep-512` 87.9263, `neighbour-delta` 81.4951, `blend-wd-nd-lgbm-rules` 88.2864
- Not included: Hamming k-NN (best 0.51983) and the Label Powerset decoder (0.23092), which exist as procedural scripts rather than model classes
- `*-rules` never scores below its base fit, as expected from deterministic additive columns: 87.8411 > 87.6543 and 88.2864 > 88.2644
- `lookup-majority` scores **0.0165%**. Unseen CRM configurations force the global modal fallback, reproducing on validation what the test-set run found
- Seed 42 only. `s2_andre_models.ipynb` shows ~1.3 points of spread between seeds, so gaps smaller than that are unresolved
- Tier C CatBoost is included as code in the section above but not executed here; its reported EMR 83.04% is below every model in the table
